In [ ]:
from ultralytics import RTDETR
import torch
import os
import time
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PATH = '/home/team2/best.pt'
SETS = {
    'Set 1': '/home/team2/sets/set1/',
    'Set 2': '/home/team2/sets/set2/',
    'Set 3': '/home/team2/sets/set3/',
    'Set 4': '/home/team2/sets/set4/',
}
IMGSZ = 640
CONF  = 0.15
# ─────────────────────────────────────────────────────────────────────────────

print(f"Device: {torch.cuda.get_device_name(0)}")
model = RTDETR(MODEL_PATH)

# Warmup — first inference is always slower due to CUDA initialization
print("Warming up GPU...")
dummy = list(model.predict(
    source='/home/team2/sets/set1/',
    imgsz=IMGSZ, conf=CONF, device=0,
    save=False, verbose=False, stream=True
))

set_stats = {}

for set_name, set_path in SETS.items():
    n_images = len(os.listdir(set_path))
    print(f"\n── {set_name} ({n_images} images) ──")

    class_counts = defaultdict(int)
    start = time.perf_counter()

    for result in model.predict(
        source=set_path,
        imgsz=IMGSZ,
        conf=CONF,
        device=0,
        save=False,        # no disk writes
        verbose=False,
        stream=True,       # stream results instead of loading all into memory
        half=True,         # FP16 inference — faster on Orin GPU
    ):
        for cls in result.boxes.cls.tolist():
            class_counts[model.names[int(cls)]] += 1

    elapsed = time.perf_counter() - start
    avg_ms = (elapsed / n_images) * 1000

    set_stats[set_name] = {
        'images': n_images,
        'avg_ms': avg_ms,
        'fps': 1000 / avg_ms,
        'counts': dict(class_counts)
    }
    print(f"  {avg_ms:.1f}ms/image → {1000/avg_ms:.1f} FPS")

# ── Summary Table ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print(f"{'':12} {'Set 1':>8} {'Set 2':>8} {'Set 3':>8} {'Set 4':>8}")
print("="*60)

for cls in ['swimmer', 'boat', 'buoy', 'jetski']:
    row = f"{cls:<12}"
    for set_name in SETS:
        count = set_stats[set_name]['counts'].get(cls, 0)
        row += f" {count:>8}"
    print(row)

print("-"*60)
fps_row = f"{'FPS':<12}"
ms_row  = f"{'ms/image':<12}"
for set_name in SETS:
    fps_row += f" {set_stats[set_name]['fps']:>8.1f}"
    ms_row  += f" {set_stats[set_name]['avg_ms']:>8.1f}"
print(fps_row)
print(ms_row)
print("="*60)

total_images = sum(s['images'] for s in set_stats.values())
total_time = sum(s['avg_ms'] * s['images'] for s in set_stats.values()) / 1000
print(f"\nOverall: {total_images} images in {total_time:.1f}s → {total_images/total_time:.1f} FPS average")